# Frequency and Keyness

<a target="_blank" href="https://colab.research.google.com/github/zentralbibliothek-zuerich/zblab-summerschool-2026/blob/main/notebooks/03_frequency_and_keyness.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook contains boilerplate code that you don't need to interact with directly.

The sections where you can safely experiment or customize are clearly marked with such comments:

```python
# ⬇️⬇️⬇️
YOUR_INPUT = ""
# ⬆️⬆️⬆️

## Setup

### Housekeeping (no interaction required)

In [ ]:
%pip install simplemma
%pip install nltk

❗ Please restart the kernel/runtime after installing the package to ensure that all changes take effect.

(Google Colab might initiate a restart on its own)

In [ ]:
import os
from pathlib import Path
from typing import Literal, Dict

import nltk
import pandas as pd
import simplemma
from tqdm import tqdm

tqdm.pandas()
nltk.download("punkt_tab")
nltk.download("stopwords")

In [ ]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')

In [ ]:
def confirm(question: str = "Do you want to execute this cell?") -> bool:
    """Ask for a yes/no confirmation in the notebook.

    Args:
        question: Prompt shown to the user.

    Returns:
        True for yes, False for no.
    """
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

### Setup (Interaction required)

In [ ]:
### ⬇️⬇️⬇️  Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
USE_YOUR_DATA = False
### ⬆️⬆️⬆️

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

## Load the data


### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [ ]:
if USE_YOUR_DATA:
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet"
    raw_df = pd.read_parquet(RAWDATA_PATH)

### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load from Github

In [ ]:
if not USE_YOUR_DATA:
    print("Loading data from GitHub...", end=" ")
    RAWDATA_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.parquet"
    raw_df = pd.read_parquet(RAWDATA_URL)
    print("Done!")

### Parsing and preparing

For easier handling, the cell below extracts only the year from the complex date column.

`1880-03-07T00:00:00+00:00` ➡️ `1880`

In [ ]:
raw_df["year"] = pd.to_datetime(raw_df["date"]).dt.year

## Preprocess Corpus

At the moment, the corpus only consists metadate and the full text as a long string of characters.
We don't know where sentences or even words begin and end.

The next cell will add such information to the corpus, a step we call *preprocessing*.

In [ ]:
if not confirm("Do you want to preprocess the corpus?"):
    raise RuntimeError("❌ Preprocessing aborted by user.")

def sentencize(s: str) -> list[str]:
    """Split a text into German sentences.

    Args:
        s: Full article text.

    Returns:
        Sentence list in original order.
    """
    sentences = nltk.tokenize.sent_tokenize(s, language="german")
    return sentences

def tokenize(s: str) -> list[str]:
    """Split one sentence into tokens.

    Args:
        s: A single sentence.

    Returns:
        Token list.
    """
    tokens = nltk.tokenize.word_tokenize(s, language="german")
    return tokens

def lemmatize(s: list[str]) -> list[str]:
    """Lemmatize a token list with German rules.

    Args:
        s: Tokens for one sentence.

    Returns:
        Lemmatized tokens.
    """
    lemmatized = [simplemma.lemmatize(word, lang="de") for word in s]
    return lemmatized


tqdm.pandas(desc="Applying sentencization")
raw_df["_sentences"] = raw_df["content"].progress_apply(sentencize)

tqdm.pandas(desc="Applying tokenization")
raw_df["tokens"] = raw_df["_sentences"].progress_apply(lambda sentences: [tokenize(sentence) for sentence in sentences])

tqdm.pandas(desc="Applying lemmatization")
raw_df["lemmas"] = raw_df["tokens"].progress_apply(lambda tokens: [lemmatize(token_list) for token_list in tokens])

### Saving the added information

We save the added information, so that we don't need to re-run preprocessing whenever we need access to tokens or lemmas.

In [ ]:
if USE_YOUR_DATA:
    raw_df[["_sentences"]].to_parquet(DATA_DIR / f"{CORPUS_NAME}.filtered.sentences.parquet")
    raw_df[["lemmas"]].to_parquet(DATA_DIR / f"{CORPUS_NAME}.filtered.lemmas.parquet")
    raw_df[["tokens"]].to_parquet(DATA_DIR / f"{CORPUS_NAME}.filtered.tokens.parquet")

### Loading existing preprocessing information

In [ ]:
if not confirm("Do you want to load precomputed tokens and lemmas?"):
    raise RuntimeError("❌ Loading precomputed tokens and lemmas aborted by user.")

if USE_YOUR_DATA:
    print("Loading data from local storage...", end=" ")
    lemma_df = pd.read_parquet(DATA_DIR / f"{CORPUS_NAME}.filtered.lemmas.parquet")
    wordform_df = pd.read_parquet(DATA_DIR / f"{CORPUS_NAME}.filtered.tokens.parquet")
    print("Done!")
else:
    print("Loading lemmas from GitHub...", end=" ")
    LEMMA_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.lemmas.parquet"
    lemma_df = pd.read_parquet(LEMMA_URL)
    print("Done!")

    print("Loading tokens from GitHub...", end=" ")
    TOKENS_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.tokens.parquet"
    wordform_df = pd.read_parquet(TOKENS_URL)
    print("Done!")

In [ ]:
if "lemmas" not in raw_df.columns or raw_df["lemmas"].isna().all():
    raw_df = raw_df.join(lemma_df)

if "tokens" not in raw_df.columns or raw_df["tokens"].isna().all():
    raw_df = raw_df.join(wordform_df)

## Frequency analyses

### Frequenzen 

Die Frequenz eines Tokens oder einer Sequenz gibt an, wie oft die jeweilige Einheit in einem Korpus vorkommt. Es gibt verschiedene Frequenzarten:
- absolute Frequenz
- relative Frequenz
- normalisierte Frequenz

So berechnen/zählen wir die verschiedenen Arten von Frequenzen:

*T* = Gesamtanzahl der Token im Korpus

| absolute Frequenz | relative Frequenz | normalisierte Frequenz |
| --- | --- | --- |
| *n* Token | *n* Token / T | *n* Token / T * 1'000'000 |

💡 Frequenzen können uns einen guten ersten Eindruck über unsere Daten geben.

Hier zählen wir zunächst absolute Frequenzen.

In [ ]:
from collections import Counter

def count_tokens(
    df: pd.DataFrame,
    column: str,
    show_progress: bool = True,
) -> Counter[str]:
    """Count token frequencies for a tokenized dataframe column.

    Args:
        df: Input dataframe with nested token lists.
        column: Column to count (usually lemmas or tokens).
        show_progress: Whether to show a progress bar.

    Returns:
        Frequency counter of tokens.
    """
    counter: Counter[str] = Counter()
    for doc in tqdm(df[column], desc="Counting Tokens", leave=False, disable=not show_progress):
        for sentence in doc:
            for token in sentence:
                counter[token] += 1
    return counter

# ⬇️⬇️⬇️   
c = count_tokens(raw_df, "lemmas") # or "tokens")
c.most_common(10)
# ⬆️⬆️⬆️

### Compare frequency across time

Nachdem wir die Frequenzen für das gesamte Korpus gezählt haben, können wir sie nun für verschiedene Arten von Subkorpora zählen. Zum Beispiel könenn wir Subkorpora bilden, indem wir das Gesamtkorpus in mehrere Zeitabschnitte unterteilen (hier jeweils 25 Jahre).

Wir zählen zunächst absolute Frequenzen und berechnen zudem relative sowie normalisierte Frequenzen. <br><br> 💡 Relative Frequenzen werden gerne verwendet, da absolute Frequenzen uns keine Informationen über den Korpuskontext liefern, in dem das Token/die Einheit gezählt wurde.


In [ ]:
import matplotlib.pyplot as plt

class FrequencyAnalysisDiachronic:
    """Analyze token frequencies across fixed year intervals."""

    def __init__(
        self,
        df: pd.DataFrame,
        column: Literal["lemmas", "tokens"],
        n_years: int,
    ) -> None:
        """Prepare interval-wise token counts.

        Args:
            df: Corpus dataframe with a year column and tokenized text.
            column: Token column to analyze.
            n_years: Width of each time interval.
        """
        self.df = df.copy()
        self.column = column

        self.min_year = self.df["year"].min()
        self.max_year = self.df["year"].max()

        self.interval_counters = self.get_frequency_by_interval(n_years)
        self.intervals = list(self.interval_counters.keys())
        self.interval_total = {
            interval: sum(counter.values())
            for interval, counter
            in self.interval_counters.items()
        }

    def get_frequency_by_interval(self, n_years: int) -> dict[str, Counter[str]]:
        """Count tokens for each year interval.

        Args:
            n_years: Width of each time interval.

        Returns:
            Mapping from year-interval labels to token counters.
        """
        if n_years <= 0:
            raise ValueError("n_years must be a positive integer.")

        # Snap start to the nearest lower multiple of n_years
        start = (self.min_year // n_years) * n_years
        stop = self.max_year + n_years

        bins = list(range(start, stop + 1, n_years))
        labels = [f"{y}-{y + n_years - 1}" for y in bins[:-1]]

        # Assign each document to a year bucket
        # With n_years = 25:
        #   1833 -> 1825 -> "1825-1849"
        #   1900 -> 1900 -> "1900-1924"
        #   1898 -> 1875 -> "1875-1899"
        self.df["year_interval"] = pd.cut(self.df["year"], bins=bins, labels=labels, include_lowest=True)

        # Calculate counts for each interval
        interval_counters: dict[str, Counter[str]] = {}
        # Filter out NaN intervals if any document year falls outside the defined bins
        for interval in tqdm(labels, desc="Iterating intervals"):
            group_df = self.df[self.df["year_interval"] == interval]
            interval_counters[interval] = count_tokens(group_df, column=self.column, show_progress=False)
        return interval_counters

    def get_abs_frequencies_for_word(self, word: str) -> dict[str, int]:
        """Return absolute frequencies of one word per interval."""
        frequencies = {}
        for interval in self.intervals:
            frequencies[interval] = self.interval_counters[interval][word]
        return frequencies

    def get_rel_frequencies_for_word(self, word: str) -> dict[str, float]:
        """Return relative frequencies of one word per interval."""
        frequencies = {}
        for interval in self.intervals:
            try:
                frequencies[interval] = self.interval_counters[interval][word] / self.interval_total[interval]
            except ZeroDivisionError:
                frequencies[interval] = 0.0
        return frequencies

    def get_norm_frequencies_for_word(self, word: str) -> dict[str, float]:
        """Return frequencies normalized to one million tokens."""
        frequencies = self.get_rel_frequencies_for_word(word)
        frequencies = {interval: freq * 1000000 for interval, freq in frequencies.items()}
        return frequencies

    def plot_frequencies(
        self,
        words: list[str],
        freqtype: Literal["absolute", "relative", "normalized"] = "relative",
    ) -> None:
        """Plot token volume and word frequencies over time.

        Args:
            words: Words to plot in the lower panel.
            freqtype: Frequency mode (absolute, relative, or normalized).
        """
        fig, (ax_total, ax_rel) = plt.subplots(
            2,
            1,
            figsize=(12, 8),
            sharex=True,
            gridspec_kw={"height_ratios": [1, 3]},
        )

        # Smaller top subplot: total tokens per interval.
        ax_total.bar(self.intervals, [self.interval_total[interval] for interval in self.intervals])
        ax_total.set_ylabel("Token Count")
        ax_total.set_title("Total Tokens per Interval")
        ax_total.grid(True, linestyle="--", alpha=0.7)

        # Larger bottom subplot: relative frequencies for selected words.
        for word in words:
            if freqtype == "absolute":
                freqs = self.get_abs_frequencies_for_word(word)
            elif freqtype == "relative":
                freqs = self.get_rel_frequencies_for_word(word)
            elif freqtype == "normalized":
                freqs = self.get_norm_frequencies_for_word(word)
            else:
                raise ValueError(f"Unknown type: {freqtype}")

            series = pd.Series(freqs)
            ax_rel.plot(series.index.astype(str), series.values, label=word, marker="o", markersize=4)

        if freqtype == "absolute":
            ax_rel.set_ylabel("Absolute Frequency")
            ax_rel.set_title("Absolute Frequencies per Interval")
        elif freqtype == "relative":
            ax_rel.set_ylabel("Relative Frequency")
            ax_rel.set_title("Relative Frequencies per Interval")
        elif freqtype == "normalized":
            ax_rel.set_ylabel("Normalized Frequency")
            ax_rel.set_title("Normalized Frequencies per Interval")

        ax_rel.tick_params(axis="x", rotation=45)
        ax_rel.grid(True, linestyle="--", alpha=0.7)
        if words:
            ax_rel.legend()

        fig.tight_layout()
        plt.show()


In [ ]:
# ⬇️⬇️⬇️
N_YEARS = 10
COLUMN = "lemmas" # or "tokens"
# ⬆️⬆️⬆️

fa_diachron = FrequencyAnalysisDiachronic(raw_df, column=COLUMN, n_years=N_YEARS)

In [ ]:
fa_diachron.plot_frequencies(["Bundesrath", "Bundesrat"], freqtype="relative")

In [ ]:
# ⬇️⬇️⬇️ Try other search terms
SEARCH_TERMS = ["Armuth", "Armut"]
FREQUENCY_TYPE = "absolute" # Choose between "absolute", "relative", and "normalized"
# ⬆️⬆️⬆️

fa_diachron.plot_frequencies(SEARCH_TERMS, freqtype=FREQUENCY_TYPE)

## Collocation Analyses

### Kollokationen 

Als Kollokationen bezeichnen wir feste Wortverbindungen. Das sind Wörter, die überzufällig häufig gemeinsam (in einem Korpus) auftreten. Wir schauen uns dazu die Wörter rechts und links eines Zielwortes (das Syntagma des Wortes) im Korpus an. Kollokationen können so statistisch berechnet werden.
Zur Berechnung können verschiedene Parameter gesetzt werden. 
1. Das Zielwort, für das die Kollokationen berechnet werden sollen
2. Das statistische Mass zur Berechnung 
3. Das sogenannte "Window", also die Spanne an Wörtern um ein Zielwort, die zur Berechnung verwendet werden sollen. Das "Window" wird in Wörtern nach links (L) und Wörtern nach rechts (R) angegeben.

### Was sagen uns Kollokationen?

Wie zuvor definiert, sind Kollokationen Wörter, die häufiger zusammen mit einem bestimmten Zielwort auftreten, als es dem Zufall entsprechen würde. 
Kollokationen sind zudem häufig sprachspezifisch. Nehmen wir folgendes Beispiel: 
Wir sagen: "Ich gehe mir die Zähne putzen." und nicht "Ich gehe mir die Zähne waschen."
*Zähne* kommt also insgesamt überzufällig häufig mit *putzen* vor, aber nicht mit *waschen*.

Dieses gemeinsame Auftreten von Wörtern ist in der Regel also ein Muster, das wir in den Daten erkennen können. Das bedeutet, dass Wörter, die gemeinsam auftreten, dies wiederholt tun, wodurch die Kollokatoren die Bedeutung (Semantik) der Zielwörter beeinflussen. Die Bedeutung wird so im wiederholten Gebrauch etabliert und manifestiert.
John Rupert Firth hat dies 1957 so formuliert: 

> „You shall know a word by the company it keeps."

💡 Kollokationen sagen uns also nicht nur, welche Wörter gemeinsam auftreten, sondern sie geben uns auch Auskunft über die Bedeutung der Zielwörter.


### Collocation code

In [ ]:
import math
from collections import Counter
from functools import lru_cache
from typing import Iterable, Literal

SMALL = 1/2

class CollocationEngine:
    """Compute collocation scores for one corpus slice."""

    def __init__(
        self,
        df: pd.DataFrame,
        column: Literal["lemmas", "tokens"],
        window: int = 5,
        stopwords: Iterable[str] = [],
    ) -> None:
        """Prepare cached token statistics for collocation scoring.

        Args:
            df: Corpus dataframe with tokenized text.
            column: Token column to use.
            window: Number of tokens left/right of a node token.
            stopwords: Tokens to mask as <STOPWORD>.
        """
        self.df = df.copy()
        self.column = column
        self.window = window
        self.stopwords = stopwords

        self.token_counts = count_tokens(self.df, column=self.column)

        stopword_count = sum(count for token, count in self.token_counts.items() if token in self.stopwords)
        for stopword in self.stopwords:
            self.token_counts.pop(stopword, None)
        self.token_counts["<STOPWORD>"] = stopword_count

        self.total_tokens = sum(self.token_counts.values())

    def calc_llr(self, data: pd.Series) -> float:
        """Calculate log-likelihood ratio for one collocate row."""
        llr_value = 0.0
        for node_presence in ["n", "xn"]:
            for collocate_presence in ["c", "xc"]:
                o = data[f"o_{node_presence}_{collocate_presence}"]
                e = data[f"e_{node_presence}_{collocate_presence}"]
                if o > 0 and e > 0:
                    llr_value += o * math.log(o / e)
        return llr_value * 2

    def calc_lr(self, data: pd.Series) -> float:
        """Calculate log-ratio for one collocate row."""
        logprob_n = math.log((data["o_n_c"] + SMALL) / (data["o_n"] + SMALL))  # Add-one smoothing
        logprob_xn = math.log((data["o_xn_c"] + SMALL) / (data["o_xn"] + SMALL))  # Add-one smoothing
        return logprob_n - logprob_xn

    def calc_mi(self, data: pd.Series) -> float:
        """Calculate mutual information for one collocate row."""
        return math.log((data["o_n_c"] + SMALL) / (data["e_n_c"] + SMALL))  # Add-one smoothing

    def get_collocations_for_word(
        self,
        node: str | Iterable[str],
        metric: Literal["llr", "lr", "mi"] = "llr",
        n: int | None = 20,
    ) -> pd.DataFrame:
        """Return scored collocations for one node or node set.

        Args:
            node: Node token or list of node tokens.
            metric: Scoring metric (llr, lr, or mi).

        Returns:
            User-facing dataframe with counts and metric values.
        """
        if isinstance(node, str):
            normalized_nodes = (node,)
        else:
            normalized_nodes = tuple(node)

        if len(normalized_nodes) == 0:
            return pd.DataFrame()

        scored_df = self._score_collocation_counts(normalized_nodes, metric)
        scored_df = scored_df.head(n) if n is not None else scored_df

        return self._reformat_df_for_user(scored_df, metric)

    def _reformat_df_for_user(self, df: pd.DataFrame, metric: Literal["llr", "lr", "mi"]) -> pd.DataFrame:
        """Select and rename technical columns for notebook output."""
        if df.empty:
            return df.copy()

        user_df = df[[
            "collocate",
            "o_n_c",
            "o_n_xc",
            "o_xn_c",
            "o_c",
            "o_n",
            "metric",

        ]].copy()
        user_df = user_df.rename(columns={
            "o_n_c": "Node with Collocate",
            "o_n_xc": "Node without Collocate",
            "o_xn_c": "Collocate without Node",
            "o_c": "Collocate Count",
            "o_n": "Node Count",
            "metric": metric,
        })
        return user_df

    def _score_collocation_counts(
        self,
        node: tuple[str, ...],
        metric: Literal["llr", "lr", "mi"],
    ) -> pd.DataFrame:
        """Score collocate counts with the selected metric."""
        colloc_df = self._get_collocation_counts_cached(node)
        if colloc_df.empty:
            return colloc_df.copy()

        scored_df = colloc_df.copy()
        if metric == "llr":
            scored_df["metric"] = scored_df.apply(self.calc_llr, axis=1)
        elif metric == "lr":
            scored_df["metric"] = scored_df.apply(self.calc_lr, axis=1)
        elif metric == "mi":
            scored_df["metric"] = scored_df.apply(self.calc_mi, axis=1)
        else:
            raise ValueError(f"Unknown metric: {metric}")

        scored_df = scored_df.sort_values(by="metric", ascending=False)
        return scored_df

    @lru_cache(maxsize=128)
    def _get_collocation_counts_cached(self, node: tuple[str, ...]) -> pd.DataFrame:
        """Build contingency-table counts for a cached node tuple."""
        columns = [
            "collocate",
            "o_n",
            "o_xn",
            "o_c",
            "o_n_c",
            "o_n_xc",
            "o_xn_c",
            "o_xn_xc",
            "e_n_c",
            "e_n_xc",
            "e_xn_c",
            "e_xn_xc",
        ]

        o_n = sum(self.token_counts.get(n, 0) for n in node)
        if o_n == 0:
            print(f"⚠️ The word '{node}' does not appear in the corpus. No collocations to compute.")
            return pd.DataFrame(columns=columns)

        if any(n in self.stopwords for n in node):
            print(f"⚠️ The word '{node}' is a stopword. Consider setting mask_stopwords=False to include it in collocation counts.")
            return pd.DataFrame(columns=columns)

        collocation_counts = Counter()

        for doc in tqdm(self.df[self.column], desc=f"Counting collocations for '{node}'", leave=False):
            full_text = [token for sentence in doc for token in sentence]
            if self.stopwords:
                full_text = [token if token not in self.stopwords else "<STOPWORD>" for token in full_text]
            for i, token in enumerate(full_text):
                if token in node:
                    start = max(i - self.window, 0)
                    end = min(i + self.window + 1, len(full_text))
                    collocates = full_text[start:i] + full_text[i + 1:end]
                    collocates = set(collocates)  # Avoid overcounting the same collocate multiple times in the same window
                    collocation_counts.update(collocates)

        rows = []
        for collocate, o_n_c in collocation_counts.items():
            o_c = self.token_counts.get(collocate, 0)

            if o_n_c > o_c:
                o_n_c = o_c

            o_n_xc = o_n - o_n_c
            o_xn_c = o_c - o_n_c
            o_xn_xc = self.total_tokens - (o_n + o_c - o_n_c)
            o_xn = self.total_tokens - o_n

            e_n_c = (o_n * o_c) / self.total_tokens
            e_n_xc = (o_n * (self.total_tokens - o_c)) / self.total_tokens
            e_xn_c = ((self.total_tokens - o_n) * o_c) / self.total_tokens
            e_xn_xc = ((self.total_tokens - o_n) * (self.total_tokens - o_c)) / self.total_tokens

            rows.append({
                "collocate": collocate,
                "o_n_c": o_n_c,
                "o_n_xc": o_n_xc,
                "o_xn_c": o_xn_c,
                "o_xn_xc": o_xn_xc,
                "o_n": o_n,
                "o_xn": o_xn,
                "o_c": o_c,
                "e_n_c": e_n_c,
                "e_n_xc": e_n_xc,
                "e_xn_c": e_xn_c,
                "e_xn_xc": e_xn_xc,
            })

        colloc_df = pd.DataFrame(rows, columns=columns)
        return colloc_df


### Analyses

### Stopwords / Stoppwörter

Stoppwörter sind Wörter, die wir in den Ergebnissen „nicht sehen wollen", da sie potenziell andere interessante Ergebnisse verdecken. Der Begriff „Wörter" ist hier etwas irreführend, da wir häufig auch andere Elemente herausfiltern, wie etwa Satzzeichen, noise (OCR-Fehler, Seitenzahlen, Header ...). Natürlich können aber auch tatsächlich Wörter gefiltert werden. Die Stoppwortliste wird in der Regel zu Beginn festgelegt, kann aber iterativ angepasst werden. <br><br>
❗️Je nach Art der gewünschten Analyse kann es sinnvoll und sehr empfehlenswert sein, Stoppwörter beizubehalten.


In [ ]:
STOPWORDS = set(nltk.corpus.stopwords.words("german"))

# ⬇️⬇️⬇️ Adjust the stopword list if you find other undesired characters or words in your analyses.
# ❗ Stopword removal is unelegant but necessary for certain collocation metrics.
# Take caution, as you are altering and biasing your object of study.
STOPWORDS = STOPWORDS.union(set([
    "»", 
    "«", 
    ".", 
    ",",
]))
COLUMN = "lemmas" # or "tokens"
WINDOW = 5
# ⬆️⬆️⬆️

colloc_engine = CollocationEngine(raw_df, column=COLUMN, window=WINDOW, stopwords=list(STOPWORDS))

In [ ]:
# ⬇️⬇️⬇️ Adjust the query here to find collocates for other words or with other metrics.
colloc_engine.get_collocations_for_word("Armut", metric="llr")
# ⬆️⬆️⬆️

In [ ]:
# ⬇️⬇️⬇️ You can combine spellings or inflectional variants in the query to get more robust results.
colloc_engine.get_collocations_for_word(["Armut", "Armuth"], metric="mi")
# ⬆️⬆️⬆️

### Compare collocations across different time slices

### Kollokationen vergleichen 

Ähnlich wie beim Vergleich der Frequenzen über die Zeit, kann dies auch mit den Kollokationen durchgeführt werden. Dazu wird das Gesamtkorpus wiederum in Subkoprora unterteilt, auf denen dann einzeln die Kollokationen für ein oder mehrere Zielwörter gerechnet werden können.

#### Code for collocation comparison class

In [ ]:
class CollocationComparer:
    """Compare collocation outputs across multiple corpus subsets."""

    def __init__(
        self,
        dfs: dict[str, pd.DataFrame],
        column: Literal["lemmas", "tokens"],
        window: int = 5,
        stopwords: Iterable[str] = [],
    ) -> None:
        """Create one collocation engine per subset.

        Args:
            dfs: Named dataframe subsets to compare.
            column: Token column used for analysis.
            window: Number of neighboring tokens per side.
            stopwords: Tokens to mask as <STOPWORD>.
        """
        self.dfs = dfs
        self.column = column
        self.window = window
        self.stopwords = stopwords

        self.engines = {
            name: CollocationEngine(df, column=column, window=window, stopwords=stopwords)
            for name, df in dfs.items()
        }

    def compare_collocations_for_word(
        self,
        node: str | list[str],
        metric: Literal["llr", "lr", "mi"] = "llr",
    ) -> pd.DataFrame:
        """Build a side-by-side top-collocate table for each subset.

        Args:
            node: One node token or a list of node variants.
            metric: Collocation metric to rank by.

        Returns:
            Combined dataframe of top collocates per subset.
        """
        colloc_dfs = {}
        for name, engine in self.engines.items():
            colloc_dfs[name] = engine.get_collocations_for_word(node, metric=metric)

        col_names = []
        col_names.append("Rank")
        for name in self.dfs.keys():
            col_names.append(f"{name}_collocate")
            col_names.append(f"{name}_{metric}")

        comparison_df = pd.DataFrame()

        for name, colloc_df in colloc_dfs.items():
            colloc_df = colloc_df.sort_values(by=metric, ascending=False).head(20)
            colloc_df = colloc_df.reset_index(drop=True)
            colloc_df = colloc_df[["collocate", "Node with Collocate"]]
            colloc_df.columns = [f"{name}_collocate", f"{name}_#"]
            comparison_df = pd.concat([comparison_df, colloc_df], axis=1)

        return comparison_df

#### Analyses

In [ ]:
# ⬇️⬇️⬇️ Define your own time intervals (or other filters) to compare different subsets of the corpus.
dfs = {
    "pre1880": raw_df[raw_df["year"] < 1880],
    "1880-1900": raw_df[(raw_df["year"] >= 1880) & (raw_df["year"] < 1900)],
    "1900-1920": raw_df[(raw_df["year"] >= 1900) & (raw_df["year"] < 1920)],
    "post1920": raw_df[raw_df["year"] >= 1920],
}

comparer = CollocationComparer(dfs, column="lemmas", window=5, stopwords=STOPWORDS)
# ⬆️⬆️⬆️

In [ ]:
# ⬇️⬇️⬇️ Adjust the node(s) and metric to compare collocates across the defined subsets.
NODES = ["Armenpflege"]
METRIC = "llr" # Choose between "llr", "lr", and "mi"
# ⬆️⬆️⬆️

comparer.compare_collocations_for_word(NODES, metric=METRIC)